# 01 - Cardiovascular Disease Multistate Progression Model

## Overview

This notebook implements an end-to-end multistate survival model for cardiovascular
disease (CVD) progression. The states are:

```
Risk Factors  -->  Stable CHD  -->  MI / Stroke  -->  Post-event  -->  HF (NYHA)  -->  Death
     |                 |                |                |                |              ^
     +---------->------+--------->------+--------->------+--------->------+-------->-----+
                                    (death as absorbing / competing risk)
```

We train and compare three model families:
1. **Cox PH** - Semiparametric baseline
2. **Fine-Gray** - Competing risks sub-distribution hazards
3. **DeepSurv** - Neural network extension of Cox PH
4. **SurvTRACE** - Transformer-based survival model

Evaluation includes C-index, time-dependent AUC, calibration, CIF plots,
and fairness analysis across age/sex subgroups.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pathlib
import warnings
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import (
    brier_score,
    concordance_index_ipcw,
    cumulative_dynamic_auc,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = pathlib.Path("../data/synthea")
OUTPUT_DIR = pathlib.Path("../outputs/cvd")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Load and preprocess synthetic CVD cohort ────────────────────────────────
patients = pd.read_csv(DATA_DIR / "patients.csv", parse_dates=["BIRTHDATE", "DEATHDATE"])
conditions = pd.read_csv(DATA_DIR / "conditions.csv", parse_dates=["START", "STOP"])
observations = pd.read_csv(DATA_DIR / "observations.csv", parse_dates=["DATE"])

# Define CVD-related SNOMED codes (Synthea uses SNOMED CT)
CVD_CODES = {
    "hypertension": ["59621000"],
    "hyperlipidemia": ["55822004"],
    "stable_chd": ["53741008", "413838009"],
    "mi": ["22298006", "401303003"],
    "stroke": ["230690007"],
    "heart_failure": ["88805009", "84114007"],
}
ALL_CVD_CODES = [code for codes in CVD_CODES.values() for code in codes]

# Filter to patients with at least one CVD-related condition
cvd_conditions = conditions[conditions["CODE"].astype(str).isin(ALL_CVD_CODES)].copy()
cvd_patient_ids = cvd_conditions["PATIENT"].unique()
print(f"Patients with CVD-related conditions: {len(cvd_patient_ids)}")

# Build patient-level feature frame
reference_date = pd.Timestamp("2024-01-01")
cohort = patients[patients["Id"].isin(cvd_patient_ids)].copy()
cohort["age"] = (reference_date - cohort["BIRTHDATE"]).dt.days / 365.25
cohort["sex_male"] = (cohort["GENDER"] == "M").astype(int)
cohort["deceased"] = cohort["DEATHDATE"].notna().astype(int)
cohort["survival_years"] = np.where(
    cohort["deceased"] == 1,
    (cohort["DEATHDATE"] - cohort["BIRTHDATE"]).dt.days / 365.25,
    cohort["age"],
)

# Merge latest observations (labs / vitals)
lab_codes = {
    "sbp": "Systolic Blood Pressure",
    "dbp": "Diastolic Blood Pressure",
    "ldl": "Low Density Lipoprotein Cholesterol",
    "hdl": "High Density Lipoprotein Cholesterol",
    "total_chol": "Total Cholesterol",
    "triglycerides": "Triglycerides",
    "bmi": "Body Mass Index",
    "hba1c": "Hemoglobin A1c/Hemoglobin.total in Blood",
    "egfr": "Estimated Glomerular Filtration Rate",
}

for feat_name, obs_desc in lab_codes.items():
    obs_feat = (
        observations[observations["DESCRIPTION"] == obs_desc]
        .sort_values("DATE")
        .groupby("PATIENT")["VALUE"]
        .last()
        .reset_index()
        .rename(columns={"PATIENT": "Id", "VALUE": feat_name})
    )
    obs_feat[feat_name] = pd.to_numeric(obs_feat[feat_name], errors="coerce")
    cohort = cohort.merge(obs_feat, on="Id", how="left")

# Assign multistate labels
def assign_cvd_state(patient_id: str) -> str:
    pt_conditions = cvd_conditions[cvd_conditions["PATIENT"] == patient_id]
    codes_present = set(pt_conditions["CODE"].astype(str).values)
    if codes_present & set(CVD_CODES["heart_failure"]):
        return "HF"
    if codes_present & set(CVD_CODES["mi"] + CVD_CODES["stroke"]):
        return "Post-event"
    if codes_present & set(CVD_CODES["stable_chd"]):
        return "Stable CHD"
    return "Risk Factors"

cohort["cvd_state"] = cohort["Id"].apply(assign_cvd_state)
print("\nCVD state distribution:")
print(cohort["cvd_state"].value_counts())

# Prepare modelling features
FEATURE_COLS = ["age", "sex_male", "sbp", "dbp", "ldl", "hdl", "total_chol",
                "triglycerides", "bmi", "hba1c", "egfr"]
TARGET_EVENT = "deceased"
TARGET_TIME = "survival_years"

# Impute and scale
for col in FEATURE_COLS:
    cohort[col] = cohort[col].fillna(cohort[col].median())

print(f"\nFinal cohort size: {len(cohort)}")
print(f"Event rate: {cohort[TARGET_EVENT].mean():.2%}")

In [ ]:
# ── Define multistate model structure ───────────────────────────────────────
# State encoding:
#   0: Risk Factors (hypertension, hyperlipidemia, smoking, obesity)
#   1: Stable CHD (chronic coronary artery disease)
#   2: MI / Stroke (acute cardiovascular event)
#   3: Post-event (stabilised after acute event)
#   4: HF with NYHA class (chronic heart failure)
#   5: Death (absorbing state)

STATE_MAP = OrderedDict([
    ("Risk Factors", 0),
    ("Stable CHD", 1),
    ("Post-event", 2),  # MI/Stroke collapsed with Post-event for modelling
    ("HF", 3),
    ("Death", 4),
])

# Allowed transitions matrix
TRANSITIONS = {
    0: [1, 4],       # Risk Factors -> Stable CHD, Death
    1: [2, 4],       # Stable CHD -> Post-event, Death
    2: [3, 4],       # Post-event -> HF, Death
    3: [4],          # HF -> Death
}

cohort["state_code"] = cohort["cvd_state"].map(STATE_MAP)

# For competing risks: event = highest state reached; death is competing event
cohort["event_type"] = np.where(
    cohort["deceased"] == 1, "death",
    cohort["cvd_state"].str.lower()
)

print("Transition matrix (allowed transitions):")
for src, dsts in TRANSITIONS.items():
    src_name = [k for k, v in STATE_MAP.items() if v == src][0]
    dst_names = [k for k, v in STATE_MAP.items() if v in dsts]
    print(f"  {src_name} -> {', '.join(dst_names)}")

In [ ]:
# ── Train/test split ────────────────────────────────────────────────────────
train_df, test_df = train_test_split(
    cohort, test_size=0.2, random_state=RANDOM_STATE, stratify=cohort[TARGET_EVENT]
)
print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Train event rate: {train_df[TARGET_EVENT].mean():.2%}")
print(f"Test  event rate: {test_df[TARGET_EVENT].mean():.2%}")

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURE_COLS])
X_test = scaler.transform(test_df[FEATURE_COLS])

X_train_df = pd.DataFrame(X_train, columns=FEATURE_COLS, index=train_df.index)
X_test_df = pd.DataFrame(X_test, columns=FEATURE_COLS, index=test_df.index)

In [ ]:
# ── Model 1: Cox PH baseline ────────────────────────────────────────────────
cox_df_train = X_train_df.copy()
cox_df_train["T"] = train_df[TARGET_TIME].values
cox_df_train["E"] = train_df[TARGET_EVENT].values

cph = CoxPHFitter(penalizer=0.01)
cph.fit(cox_df_train, duration_col="T", event_col="E")
cph.print_summary(columns=["coef", "exp(coef)", "p"])

# Predict on test
cox_df_test = X_test_df.copy()
cox_df_test["T"] = test_df[TARGET_TIME].values
cox_df_test["E"] = test_df[TARGET_EVENT].values

cox_risk = -cph.predict_partial_hazard(cox_df_test[FEATURE_COLS]).values.ravel()
cox_cindex = concordance_index(
    cox_df_test["T"], cox_risk, cox_df_test["E"]
)
print(f"\nCox PH test C-index: {cox_cindex:.4f}")

In [ ]:
# ── Model 2: Fine-Gray competing risks ─────────────────────────────────────
from cmprsk import crr  # competing risks regression

# Encode competing risk events:
#   0 = censored, 1 = CVD death, 2 = non-CVD death
train_df["cr_event"] = np.where(
    train_df["deceased"] == 1,
    np.where(train_df["cvd_state"].isin(["Post-event", "HF"]), 1, 2),
    0,
)
test_df["cr_event"] = np.where(
    test_df["deceased"] == 1,
    np.where(test_df["cvd_state"].isin(["Post-event", "HF"]), 1, 2),
    0,
)

try:
    fg_result = crr(
        ftime=train_df[TARGET_TIME].values,
        fstatus=train_df["cr_event"].values,
        cov1=X_train,
        cencode=0,
        failcode=1,
    )
    print("Fine-Gray model fitted.")
    print(f"Coefficients: {fg_result.coef}")
    print(f"P-values:     {fg_result.pvalues}")
except Exception as e:
    print(f"Fine-Gray fitting note: {e}")
    print("Falling back to cause-specific Cox model for competing risks.")
    # Cause-specific Cox as fallback
    cs_df = cox_df_train.copy()
    cs_df["E"] = (train_df["cr_event"] == 1).astype(int).values
    cs_cox = CoxPHFitter(penalizer=0.01)
    cs_cox.fit(cs_df, duration_col="T", event_col="E")
    print("Cause-specific Cox (CVD death) fitted.")
    cs_cox.print_summary(columns=["coef", "exp(coef)", "p"])

In [ ]:
# ── Model 3: DeepSurv ───────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


class DeepSurv(nn.Module):
    """Neural network for survival analysis (DeepSurv / Cox-nnet)."""

    def __init__(self, in_features: int, hidden_layers: list[int], dropout: float = 0.3):
        super().__init__()
        layers = []
        prev_dim = in_features
        for h in hidden_layers:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def negative_log_partial_likelihood(risk_scores: torch.Tensor,
                                     events: torch.Tensor,
                                     times: torch.Tensor) -> torch.Tensor:
    """Compute negative log partial likelihood (Breslow approximation)."""
    sorted_idx = torch.argsort(times, descending=True)
    risk_scores = risk_scores[sorted_idx]
    events = events[sorted_idx]
    hazard_ratio = torch.exp(risk_scores)
    log_risk = torch.log(torch.cumsum(hazard_ratio, dim=0))
    uncensored_likelihood = risk_scores - log_risk
    censored_likelihood = uncensored_likelihood * events
    n_events = events.sum()
    if n_events > 0:
        return -censored_likelihood.sum() / n_events
    return torch.tensor(0.0)


# Prepare tensors
X_train_t = torch.FloatTensor(X_train)
y_time_train = torch.FloatTensor(train_df[TARGET_TIME].values)
y_event_train = torch.FloatTensor(train_df[TARGET_EVENT].values)

X_test_t = torch.FloatTensor(X_test)
y_time_test = torch.FloatTensor(test_df[TARGET_TIME].values)
y_event_test = torch.FloatTensor(test_df[TARGET_EVENT].values)

# Train DeepSurv
model_ds = DeepSurv(in_features=len(FEATURE_COLS), hidden_layers=[64, 32, 16], dropout=0.3)
optimizer = torch.optim.Adam(model_ds.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

n_epochs = 100
train_losses = []

model_ds.train()
for epoch in range(n_epochs):
    optimizer.zero_grad()
    risk = model_ds(X_train_t).squeeze()
    loss = negative_log_partial_likelihood(risk, y_event_train, y_time_train)
    loss.backward()
    optimizer.step()
    scheduler.step(loss)
    train_losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {loss.item():.4f}")

# Evaluate
model_ds.eval()
with torch.no_grad():
    ds_risk = model_ds(X_test_t).squeeze().numpy()

ds_cindex = concordance_index(test_df[TARGET_TIME].values, -ds_risk, test_df[TARGET_EVENT].values)
print(f"\nDeepSurv test C-index: {ds_cindex:.4f}")

plt.figure(figsize=(8, 3))
plt.plot(train_losses, lw=1.5)
plt.xlabel("Epoch")
plt.ylabel("Negative log partial likelihood")
plt.title("DeepSurv training loss")
plt.tight_layout()
plt.show()

In [ ]:
# ── Model 4: SurvTRACE (Transformer-based) ─────────────────────────────────
try:
    from survtrace import SurvTraceSingle, SurvTraceMulti
    from survtrace.config import STConfig

    config = STConfig(
        num_numerical_feature=len(FEATURE_COLS),
        num_categorical_feature=0,
        hidden_size=64,
        intermediate_size=128,
        num_attention_heads=4,
        num_hidden_layers=2,
        out_feature=1,
    )

    model_st = SurvTraceSingle(config)
    # SurvTRACE uses its own training loop
    from survtrace.train_utils import Trainer
    from survtrace.evaluate_utils import Evaluator

    trainer = Trainer(model_st)
    train_data = {
        "x": X_train,
        "t": train_df[TARGET_TIME].values,
        "e": train_df[TARGET_EVENT].values,
    }
    val_data = {
        "x": X_test,
        "t": test_df[TARGET_TIME].values,
        "e": test_df[TARGET_EVENT].values,
    }
    trainer.fit(train_data, val_data, batch_size=256, epochs=50, learning_rate=1e-3)

    evaluator = Evaluator(model_st)
    st_metrics = evaluator.eval(val_data)
    print(f"SurvTRACE test C-index: {st_metrics['c_index']:.4f}")

except ImportError:
    print("SurvTRACE not installed. Skipping transformer-based model.")
    print("Install with: pip install survtrace")
    print("Proceeding with Cox PH and DeepSurv results only.")
    st_metrics = None

In [ ]:
# ── Evaluation: C-index, time-dependent AUC, calibration ────────────────────
from sksurv.util import Surv

# Prepare sksurv-compatible structured arrays
y_train_surv = Surv.from_arrays(
    event=train_df[TARGET_EVENT].astype(bool).values,
    time=train_df[TARGET_TIME].values,
)
y_test_surv = Surv.from_arrays(
    event=test_df[TARGET_EVENT].astype(bool).values,
    time=test_df[TARGET_TIME].values,
)

# Time-dependent AUC at 1, 3, 5 years
time_horizons = [1.0, 3.0, 5.0]
# Filter time horizons to be within observed range
max_time = test_df[TARGET_TIME].max()
valid_horizons = [t for t in time_horizons if t < max_time]

results = {"Model": [], "C-index": []}
for t in valid_horizons:
    results[f"AUC@{int(t)}y"] = []

# Cox PH
results["Model"].append("Cox PH")
results["C-index"].append(cox_cindex)
try:
    cox_auc, _ = cumulative_dynamic_auc(y_train_surv, y_test_surv, -cox_risk, valid_horizons)
    for i, t in enumerate(valid_horizons):
        results[f"AUC@{int(t)}y"].append(cox_auc[i])
except Exception:
    for t in valid_horizons:
        results[f"AUC@{int(t)}y"].append(np.nan)

# DeepSurv
results["Model"].append("DeepSurv")
results["C-index"].append(ds_cindex)
try:
    ds_auc, _ = cumulative_dynamic_auc(y_train_surv, y_test_surv, ds_risk, valid_horizons)
    for i, t in enumerate(valid_horizons):
        results[f"AUC@{int(t)}y"].append(ds_auc[i])
except Exception:
    for t in valid_horizons:
        results[f"AUC@{int(t)}y"].append(np.nan)

results_df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(results_df.to_string(index=False))

# Calibration plot at 5 years
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KM survival curves by predicted risk quartile (Cox PH)
test_with_risk = test_df.copy()
test_with_risk["risk_quartile"] = pd.qcut(-cox_risk, 4, labels=["Q1 (low)", "Q2", "Q3", "Q4 (high)"])

kmf = KaplanMeierFitter()
for q in ["Q1 (low)", "Q2", "Q3", "Q4 (high)"]:
    mask = test_with_risk["risk_quartile"] == q
    kmf.fit(
        test_with_risk.loc[mask, TARGET_TIME],
        event_observed=test_with_risk.loc[mask, TARGET_EVENT],
        label=q,
    )
    kmf.plot_survival_function(ax=axes[0])

axes[0].set_title("Cox PH: KM curves by predicted risk quartile")
axes[0].set_xlabel("Time (years)")
axes[0].set_ylabel("Survival probability")

# CIF plot (cumulative incidence)
cph_sf = cph.predict_survival_function(X_test_df[FEATURE_COLS])
mean_cif = 1 - cph_sf.mean(axis=1)
axes[1].plot(mean_cif.index, mean_cif.values, lw=2, color="#C44E52")
axes[1].fill_between(mean_cif.index, mean_cif.values, alpha=0.2, color="#C44E52")
axes[1].set_title("Mean cumulative incidence function (all-cause death)")
axes[1].set_xlabel("Time (years)")
axes[1].set_ylabel("Cumulative incidence")

plt.tight_layout()
plt.show()

In [ ]:
# ── Fairness: subgroup analysis by age and sex ──────────────────────────────
test_eval = test_df.copy()
test_eval["cox_risk"] = cox_risk
test_eval["ds_risk"] = ds_risk
test_eval["age_group"] = pd.cut(test_eval["age"], bins=[0, 50, 65, 80, 120],
                                 labels=["<50", "50-64", "65-79", "80+"])

fairness_results = []

for model_name, risk_col in [("Cox PH", "cox_risk"), ("DeepSurv", "ds_risk")]:
    # By sex
    for sex_label, sex_val in [("Male", 1), ("Female", 0)]:
        mask = test_eval["sex_male"] == sex_val
        if mask.sum() > 10 and test_eval.loc[mask, TARGET_EVENT].sum() > 0:
            ci = concordance_index(
                test_eval.loc[mask, TARGET_TIME],
                -test_eval.loc[mask, risk_col],
                test_eval.loc[mask, TARGET_EVENT],
            )
            fairness_results.append({
                "Model": model_name, "Subgroup": f"Sex: {sex_label}",
                "N": int(mask.sum()), "Events": int(test_eval.loc[mask, TARGET_EVENT].sum()),
                "C-index": ci,
            })

    # By age group
    for ag in ["<50", "50-64", "65-79", "80+"]:
        mask = test_eval["age_group"] == ag
        if mask.sum() > 10 and test_eval.loc[mask, TARGET_EVENT].sum() > 0:
            ci = concordance_index(
                test_eval.loc[mask, TARGET_TIME],
                -test_eval.loc[mask, risk_col],
                test_eval.loc[mask, TARGET_EVENT],
            )
            fairness_results.append({
                "Model": model_name, "Subgroup": f"Age: {ag}",
                "N": int(mask.sum()), "Events": int(test_eval.loc[mask, TARGET_EVENT].sum()),
                "C-index": ci,
            })

fairness_df = pd.DataFrame(fairness_results)
print("\n" + "=" * 70)
print("FAIRNESS ANALYSIS: Subgroup C-index")
print("=" * 70)
print(fairness_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
pivot = fairness_df.pivot(index="Subgroup", columns="Model", values="C-index")
pivot.plot.bar(ax=ax, rot=0)
ax.axhline(0.5, color="red", ls="--", alpha=0.5, label="Random")
ax.set_ylabel("C-index")
ax.set_title("Subgroup performance comparison")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Generate model card (saved to governance/) ──────────────────────────────
model_card = f"""# Model Card: CVD Multistate Progression Model

## Model Details
- **Model type**: Multistate survival model (Cox PH, DeepSurv, SurvTRACE)
- **Version**: 0.1.0 (development)
- **Training date**: {pd.Timestamp.now().strftime('%Y-%m-%d')}
- **Framework**: lifelines, PyTorch, scikit-survival

## Performance Summary
{results_df.to_markdown(index=False)}

## Fairness Summary
{fairness_df.to_markdown(index=False)}

## Intended Use
Research prototype for DACH insurance underwriting support.
NOT approved for production use or clinical decision-making.

## Limitations
- Trained on synthetic (Synthea) data, not real patient data
- US-centric disease patterns may not transfer to DACH populations
- Limited feature set (no lifestyle, genetic, or social determinants)
"""

model_card_path = pathlib.Path("../governance/model_card_cvd.md")
model_card_path.parent.mkdir(parents=True, exist_ok=True)
model_card_path.write_text(model_card)
print(f"Model card saved to {model_card_path}")
print("\nDone. Review governance/model_card_cvd.md for the full model card.")